# Mini FSD training on Colab

Thin by design (ADR 0007): this notebook mounts Drive, installs the repo at a
pinned commit, unpacks the data and calls a `minifsd.train` module. All the
logic is in the repo, tested locally.

Before the first run, upload the packs from `data/packs/` (made by
`python -m minifsd.train.pack`) to `My Drive/mini-fsd/packs/`.

Runtime > Change runtime type > T4 GPU. If the session drops, run all cells
again: training resumes from the last checkpoint on Drive.

In [ ]:
# Settings for this run. COMMIT pins the code, so a result can be traced
# back to it; use the commit you pushed last (git rev-parse --short HEAD).
COMMIT = "main"
PACKS = ["v9.zip", "coco-people.zip"]
DRIVE = "/content/drive/MyDrive/mini-fsd"
RUN = "smoke"  # checkpoints and logs go to DRIVE/runs/RUN

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# The repo at the pinned commit. Extra packages, pinned, are added here per
# experiment and recorded in ADR 0007.
!pip install -q "git+https://github.com/SegunAdewola/mini-fsd@{COMMIT}"

In [ ]:
import pathlib
import shutil

from minifsd.train import pack

data = pathlib.Path("/content/data")
data.mkdir(exist_ok=True)
for name in PACKS:
    local = data / name
    if not local.exists():
        shutil.copy(f"{DRIVE}/packs/{name}", local)
    folder = data / name.removesuffix(".zip")
    if not folder.exists():
        print("unpacked", pack.unpack(local, data))

## Run

The training command for each experiment goes here (E2a first). It reads
from `/content/data` and writes checkpoints to `DRIVE/runs/RUN`, resuming if
any are there.